In [13]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_openai import ChatOpenAI
from typing import TypedDict
# from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
import os
import getpass
load_dotenv()

True

### Gettting all the components ready
> Document loader

> Splitter

> Embedding model

> Vector DB 

> LLM Model 

In [3]:
def doc_loader(path):
    try:
        loader = DirectoryLoader(path,
                                glob="**/*.pdf",
                                loader_cls=PyMuPDFLoader,
                                show_progress=True)
        documents = loader.load()
        print(f"Loaded {len(documents)} documents from {path}")
        return documents
    except Exception as e:
        print(f"Error loading documents from {path}: {e}")
        return None

def text_splitter(documents):
    print("Splitting documents into chunks...")
    try:
        if not documents:
            raise ValueError("No documents to split")
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, 
                                              chunk_overlap=150,
                                              length_function=len,
                                              separators=["\n\n", "\n", " ", ""])
        chunks = splitter.split_documents(documents)
        print(f"""Split into {len(chunks)} chunks
            Document splitting complete""")
        return chunks
    except Exception as e:
        print(f"Error splitting documents: {e}")
        return None

def create_vector(chunks):
    print('Loading embedding model...')
    embedding = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
    print('Creating vector store...')
    vector_store = Chroma.from_documents(chunks, 
                                         embedding, 
                                         collection_name="pdf_docs")
    print('Vector store created successfully')
    return vector_store

def llm_model():
    print('Loading LLM model...')
    llm = ChatOpenAI(
        model="moonshotai/kimi-k2.6:free", 
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1")
    return llm

### Assembeling the components

In [ ]:
print('LLM Checking the query if any query reformulation is needed...')
def rewrite_query(llm,query,state):
    llm = llm_model()
    query = state["query"]
    prompt = f"""
        Rewrite the question only if necessary for document retrieval.
        Preserve the original meaning.
        Return only the rewritten query.
        Question: {query}
"""

    rewritten_query = llm.invoke(prompt)
    return rewritten_query.content

print('Starting generation process...')
def generation():
    llm = llm_model()
    documents = doc_loader("../data")
    chunks = text_splitter(documents)
    vector_store = create_vector(chunks)
    retriever = vector_store.as_retriever(
        search_kwargs={"k": 3}
        )
    
    user_query = input("Enter your query: ")
    rewritten_query = rewrite_query(llm,user_query)
    print(f"Rewritten query: {rewritten_query}")
    results = retriever.invoke(rewritten_query)

    context = "\n".join(doc.page_content for doc in results)
    final_prompt = PromptTemplate.from_template(
        """
        Use the following context to answer the question: 
        Context : {context}
        Question: {query}
        """
    )
    final_response = llm.invoke(
        final_prompt.format(
            context=context, 
            query=rewritten_query
            )
        )
    
    print(f"Final response: {final_response.content}")
if __name__ == "__main__":
    generation()

### Lang-graph works with a shared state

In [4]:
class AgentState(TypedDict):
    query:str
    rewritten_query:str
    documents:list
    web_results:str
    answer:str

### Retrieval Node

In [ ]:
print('LLM Checking the query if any query reformulation is needed...')
def rewrite_query(llm,query,state):
    llm = llm_model()
    query = state["query"]
    prompt = f"""
        Rewrite the question only if necessary for document retrieval.
        Preserve the original meaning.
        Return only the rewritten query.
        Question: {query}
"""

    rewritten_query = llm.invoke(prompt)
    return rewritten_query.content

def retriever_documents(state):
    query=state["query"]
    retriever = create_vector.as_retriever(
        search_kwargs={"k": 3}
        )
    docs = retriever.invoke(query)

    return {
        "documents": docs
    }


# Relevence Grader agent
print("Grading the relevent documents")
def grade_documents(state):
    docs = state["documents"]
    query = state["query"]
    llm = llm_model()
    context = "\n\n".join(
        doc.page_content 
        for doc in docs
        )
    prompt = f"""
            Query:{query}
            
            Context:{context}

            Are this documents relevant?

            Answer only:
            Yes 
            No
            """
    
    result = llm.invoke(prompt)

    if "yes" in result.content.lower():
        return "Generated"
    return "Web-Search"

### Web-Search Node

In [ ]:
load_dotenv()
search_tool = TavilySearch(
    max_results=5
)
print('Loading web-search tool..')
def web_search(state):
    query = state["rewritten_query"]
    result = search_tool.invoke(query)
    return {
        "web-search":str(result)
    }
print('Web-Search tool loaded')

### Answer Generation 

In [ ]:
print('Starting Generation')
def generation(state):
    docs = state["documents"]
    web_search = state.get["web_results"]
    context = "\n\n".join(doc.page_content
                          for doc in docs)
    
    prompt = f"""
        Answer using avilable information
        Document Context:{context}
        web Content: {web_search}
        question:{state['query']}
        """
    llm = llm_model() 
    response = llm.invoke(prompt)

    return {
        "answer":response.content
    }